### Setup and study list

In [11]:
from pathlib import Path
import pandas as pd
import re, time

DATA_DIR = Path(r"C:\Users\user\OneDrive - University of Leeds\Dissertation Data\SEE AQ Projects-PURPLEAIR - sensor_data")
PROCESSED = Path.cwd().parent / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES = Path.cwd().parent / "outputs" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

# the 48 sensors we decided to keep
study = pd.read_csv(OUT_TABLES / "study_sensors.csv")
print(f"Study sensors to combine: {len(study)}")
study.head()

Study sensors to combine: 48


,sensor_id,sensor_name,location_type,lat,lon
0,15,SL001 Sunnyview Terrace,outdoor,53.774956,-1.565842
1,27,SL003 - Corn Exchange Cabinet,outdoor,53.796494,-1.540428
2,26,SL005 Kirkstall Valley Primary,outdoor,53.808520,-1.586729
3,47,SL006 - Primrose Hill Primary,outdoor,53.801888,-1.667109
4,45,SL007 Ninelands Primary,outdoor,53.789272,-1.377445


### Match each study sensor to its folder on disk

In [12]:
# Folders on disk that actually contain daily CSVs
def is_daily(name): return re.fullmatch(r"\d{4}-\d{2}-\d{2}\.csv", name) is not None

disk_folders = []
for item in sorted(DATA_DIR.iterdir()):
    if item.is_dir() and any(is_daily(p.name) for p in item.rglob("*.csv")):
        disk_folders.append(item.name)

print(f"Folders on disk with daily data: {len(disk_folders)}")

# Match by the SLxxx code, which appears in both the summary name and the folder name.
def sl_code(text):
    m = re.search(r"SL[_ ]?0*(\d+)", str(text).upper())
    return f"SL{int(m.group(1)):03d}" if m else None

study["code"] = study["sensor_name"].apply(sl_code)
folder_by_code = {}
for f in disk_folders:
    c = sl_code(f)
    if c: folder_by_code[c] = f

study["folder"] = study["code"].map(folder_by_code)

matched = study.dropna(subset=["folder"])
unmatched = study[study["folder"].isna()]
print(f"Matched to a folder: {len(matched)}")
print(f"NOT matched (no folder found): {len(unmatched)}")
if len(unmatched):
    print(unmatched[["sensor_name", "code"]].to_string(index=False))

Folders on disk with daily data: 72
Matched to a folder: 47
NOT matched (no folder found): 1
    sensor_name  code
SL71 - Kentmere SL071


###  The combine function (reads one sensor, returns data + stats)

In [13]:
def combine_one_sensor(folder_name: str):
    """Stack all daily CSVs for one sensor. Returns (dataframe, stats dict)."""
    folder = DATA_DIR / folder_name
    daily_files = sorted(p for p in folder.rglob("*.csv") if is_daily(p.name))

    frames, empty_days, bad_files = [], 0, 0
    for f in daily_files:
        try:
            d = pd.read_csv(f)
            if len(d) == 0:
                empty_days += 1
            else:
                frames.append(d)
        except Exception:
            bad_files += 1

    if not frames:
        return pd.DataFrame(), {
            "folder": folder_name, "rows": 0, "days_with_data": 0,
            "empty_days": empty_days, "bad_files": bad_files,
            "start": None, "end": None,
        }

    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"], format="ISO8601", errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)
    df.insert(0, "sensor", folder_name)

    stats = {
        "folder": folder_name,
        "rows": len(df),
        "days_with_data": len(frames),
        "empty_days": empty_days,
        "bad_files": bad_files,
        "start": df["date"].min(),
        "end": df["date"].max(),
    }
    return df, stats

In [14]:
inventory = []
t0 = time.time()
folders_to_do = matched["folder"].tolist()

for i, fname in enumerate(folders_to_do, 1):
    print(f"[{i}/{len(folders_to_do)}] {fname} ...", end=" ")
    df, stats = combine_one_sensor(fname)
    if df.empty:
        print("no data")
    else:
        df.to_parquet(PROCESSED / f"{fname}.parquet", index=False)
        print(f"{stats['rows']:,} rows  ({stats['days_with_data']} days, "
              f"{stats['empty_days']} empty)")
    inventory.append(stats)

print(f"\nDone in {time.time()-t0:.0f}s")

[1/47] SL001_Sunnyview_Terrace ... 

KeyboardInterrupt: 

In [ ]:
inv = pd.DataFrame(inventory)
# attach the sensor names + locations from the study list
inv = inv.merge(matched[["folder", "sensor_name", "lat", "lon"]],
                on="folder", how="left")

# coverage %: days with data vs. total span in days
inv["span_days"] = (pd.to_datetime(inv["end"]) - pd.to_datetime(inv["start"])).dt.days + 1
inv["coverage_pct"] = (inv["days_with_data"] / inv["span_days"] * 100).round(1)

cols = ["sensor_name", "folder", "rows", "days_with_data", "empty_days",
        "span_days", "coverage_pct", "start", "end", "bad_files"]
inv = inv[cols].sort_values("sensor_name")

inv.to_csv(OUT_TABLES / "sensor_inventory.csv", index=False)
print("Saved -> outputs/tables/sensor_inventory.csv")
inv